In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [2]:
# select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [3]:
# clone YOLOv3 implemementation
!git clone https://github.com/Lornatang/YOLOv3-PyTorch.git

Cloning into 'YOLOv3-PyTorch'...
remote: Enumerating objects: 3667, done.
remote: Counting objects: 100% (1782/1782), done.
remote: Compressing objects: 100% (597/597), done.
remote: Total 3667 (delta 1230), reused 1575 (delta 1154), pack-reused 1885 (from 1)
Receiving objects: 100% (3667/3667), 1.39 MiB | 24.07 MiB/s, done.
Resolving deltas: 100% (2454/2454), done.
Filtering content: 100% (4/4), 572.56 KiB | 215.00 KiB/s, done.


In [ ]:
!ls YOLOv3-PyTorch

In [4]:
# install YOLOv3
!ln -s YOLOv3-PyTorch/yolov3_pytorch yolov3_pytorch
!ln -s YOLOv3-PyTorch/yolov3_pytorch/tools yolov3_pytorch/tools
!ln -s YOLOv3-PyTorch/yolov3_pytorch/configs yolov3_pytorch/configs
!ln -s YOLOv3-PyTorch/yolov3_pytorch/model_configs yolov3_pytorch/model_configs

In [5]:
!pip install thop

In [ ]:
# take images, e.g. using https://imageonline.io/take-photo/

In [ ]:
# annotate images, using https://www.makesense.ai/ add label / select ROI + label / action / export / yolo

In [6]:
# download train and test images with annotations
!wget http://www.agentspace.org/download/watch-annotated.zip
!unzip watch-annotated.zip
!rm watch-annotated.zip

--2025-11-23 00:21:25--  http://www.agentspace.org/download/watch-annotated.zip
Resolving www.agentspace.org (www.agentspace.org)... 62.168.101.9
Connecting to www.agentspace.org (www.agentspace.org)|62.168.101.9|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.agentspace.org/download/watch-annotated.zip [following]
--2025-11-23 00:21:26--  https://www.agentspace.org/download/watch-annotated.zip
Connecting to www.agentspace.org (www.agentspace.org)|62.168.101.9|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1576196 (1.5M) [application/zip]
Saving to: ‘watch-annotated.zip’

watch-annotated.zip 100%[===================>]   1.50M  1.14MB/s    in 1.3s    

2025-11-23 00:21:29 (1.14 MB/s) - ‘watch-annotated.zip’ saved [1576196/1576196]

Archive:  watch-annotated.zip
   creating: data/
   creating: data/custom/
 extracting: data/custom/custom.names  
   creating: data/custom/images/
   creating: data/custom/image

In [10]:
# download pretrained model
!wget -O YOLOv3_Tiny-VOC-20231107.pth.tar https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF

--2025-11-23 00:27:14--  https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF
Resolving drive.google.com (drive.google.com)... 142.250.4.101, 142.250.4.139, 142.250.4.100, ...
Connecting to drive.google.com (drive.google.com)|142.250.4.101|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF/ [following]
--2025-11-23 00:27:16--  https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF/
Reusing existing connection to drive.google.com:443.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF/edit [following]
--2025-11-23 00:27:16--  https://drive.google.com/file/d/1J9Cqe_H0htheulX1JbjaN-SMnPrFJIoF/edit
Reusing existing connection to drive.google.com:443.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘YOLOv3_Tiny-VOC-20231107.pth.tar’

YO

In [ ]:
!cp -v

In [ ]:
from yolov3_pytorch.utils import scale_coords, xyxy2xywh, non_max_suppression, plot_one_box
from yolov3_pytorch.data.data_augment import letterbox

In [ ]:
# load the model
import yolov3_pytorch
model_path = 'watch_detector_yolov3_tiny.pth'
model = torch.load(model_path, weights_only=False).to(device)
model.eval()

In [ ]:
names = ['watch']

In [ ]:
frame = cv2.imread("data/custom/images/test/000011.jpg")
frame.shape

In [ ]:
# preprocessing
img_size = 416
img, _, _ = letterbox(frame,new_shape=img_size)
img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
blob = cv2.dnn.blobFromImage(img,1.0/255)
blob = torch.tensor(blob)
blob = blob.to(device)
blob.shape

In [ ]:
# inference
with torch.no_grad():
    output, _ = model(blob, False)

output.shape

In [ ]:
output[0,0] # x1, y1, x2, y2, objectness, probability of class 1

In [ ]:
# postprocessing - non-maximum supression
conf_thresh = 0.08
iou_thresh = 0.45
detections = non_max_suppression(output, conf_thresh, iou_thresh)[0]

In [ ]:
detections.shape

In [ ]:
detections[:10] # x1, y1, x2, y2, confidence, classid

In [ ]:
# postprocessing - rescaling
detections[:, :4] = scale_coords(blob.shape[2:], detections[:, :4], frame.shape).round()

In [ ]:
detections[:10] # x1, y1, x2, y2, confidence, classid

In [ ]:
# visualization
for detection in detections:
    *xyxy, confidence, classid  = detection
    print(f'{confidence.item():.2f},{int(xyxy[0].item())},{int(xyxy[1].item())},{int(xyxy[2].item())},{int(xyxy[3].item())},{names[classid.int().item()]}')
    plot_one_box(xyxy, frame, label=names[classid.int().item()], color=(0,0,255))

In [ ]:
plt.imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))

In [ ]:
# upload video
from google.colab import files
uploaded = files.upload()
videofile = list(uploaded.keys())[0]
print(videofile)

In [ ]:
# process image
def process_image(frame):
    img_size = 416
    img, _, _ = letterbox(frame,new_shape=img_size)
    img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    blob = cv2.dnn.blobFromImage(img,1.0/255)
    blob = torch.tensor(blob)
    blob = blob.to(device)
    with torch.no_grad():
        output, _ = model(blob, False)
    conf_thresh = 0.08
    iou_thresh = 0.45
    detections = non_max_suppression(output, conf_thresh, iou_thresh)[0]
    if detections is None:
        return frame
    detections[:, :4] = scale_coords(blob.shape[2:], detections[:, :4], frame.shape).round()
    disp = np.copy(frame)
    for detection in detections:
        *xyxy, confidence, classid  = detection
        plot_one_box(xyxy, disp, label=names[classid.int().item()], color=(0,0,255))
    return disp

In [ ]:
# process video
resultfile = 'result.avi'
video = cv2.VideoCapture(videofile)
fps = video.get(cv2.CAP_PROP_FPS)
hasFrame, frame = video.read()
out = cv2.VideoWriter()
out.open(resultfile,cv2.VideoWriter_fourcc('M','J','P','G'),fps,(frame.shape[1],frame.shape[0]))
while True:
    result = process_image(frame)
    out.write(result)
    hasFrame, frame = video.read()
    if not hasFrame:
        break
out.release()
cv2.destroyAllWindows()

In [ ]:
# download video
files.download(resultfile)